In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [2]:
!pip install trl fastchat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 kB 15.6 MB/s eta 0:00:00


In [3]:
! cd /content/drive/MyDrive/CAISI_phase2/safe-finetuning/SaLoRA/

In [4]:
! pwd

/content


In [5]:
import sys

In [6]:
sys.path.append("/content/drive/MyDrive/CAISI_phase2/safe-finetuning/SaLoRA")

In [7]:
import process_lora

In [8]:
from lowrank_prune.lib import data as data_lib

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os
# from lowrank_prune.lib.model_wrapper_low import *
# from lowrank_prune.lib.data import get_loaders
from peft import LoraConfig, get_peft_model
import time

from trl import SFTTrainer, SFTConfig
from datasets import load_dataset, Dataset
from fastchat.conversation import get_conv_template
from tqdm import tqdm

# from process_lora import process_checkpoint
import logging
import datasets
import csv

from box import Box

In [10]:
import importlib

In [11]:
importlib.reload(data_lib)

<module 'lowrank_prune.lib.data' from '/content/drive/MyDrive/CAISI_phase2/safe-finetuning/SaLoRA/lowrank_prune/lib/data.py'>

In [12]:
model_path = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [13]:
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_path)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [14]:
device = model.device

In [15]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

In [16]:
train_data_path = '/content/gdrive/MyDrive/CAISI_phase2/safe-finetuning/ft_datasets/alpaca_clean_sample_5k/train'
align_data_path = '/content/gdrive/MyDrive/CAISI_phase2/safe-finetuning/SaLoRA/lowrank_prune/data/align_train.csv'

In [17]:
dataloader = data_lib.get_loaders(
        "alpaca_train",
        nsamples=128,
        seed=0,
        seqlen=4096,
        tokenizer=tokenizer,
        disentangle=True,
    )

In [18]:
len(dataloader[0][0])
tokenizer.decode(dataloader[1][0][0])
keep_ids = torch.ne(dataloader[1][1][0], -100)
keep_ids
tokenizer.decode(dataloader[1][1][0][keep_ids])

'10 lbs is equivalent to 4.54 kilograms.'

In [19]:
dataloader_safe = data_lib.get_loaders(
        'align_short',
        nsamples=128,
        seed=0,
        seqlen=4096,
        tokenizer=tokenizer,
        disentangle=True,
    )

Generating train split: 0 examples [00:00, ? examples/s]

In [20]:
len(dataloader_safe)

128

In [21]:
# tokenizer.decode(dataloader_safe[1][0][0])
# keep_ids = torch.ne(dataloader_safe[1][1][0], -100)
# tokenizer.decode(dataloader_safe[1][1][0][keep_ids])
# messages = [
#     {"role": "system", "content": "You are a friendly chatbot who always responds in the style of a pirate",},
#     {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
#  ]
# tokenizer.apply_chat_template(messages, tokenize=False)

In [22]:
from lowrank_prune.lib.model_wrapper_low import *

In [23]:
model.config.num_hidden_layers

32

In [24]:
model = make_Act(model, verbose=False)

In [25]:
model.requires_grad_(False)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): ActLinear(
            (base): Linear(in_features=4096, out_features=4096, bias=False)
          )
          (k_proj): ActLinear(
            (base): Linear(in_features=4096, out_features=1024, bias=False)
          )
          (v_proj): ActLinear(
            (base): Linear(in_features=4096, out_features=1024, bias=False)
          )
          (o_proj): ActLinear(
            (base): Linear(in_features=4096, out_features=4096, bias=False)
          )
        )
        (mlp): LlamaMLP(
          (gate_proj): ActLinear(
            (base): Linear(in_features=4096, out_features=14336, bias=False)
          )
          (up_proj): ActLinear(
            (base): Linear(in_features=4096, out_features=14336, bias=False)
          )
          (down_proj): ActLinear(
            (base): Linea

In [26]:
clear_act_buffer(model)

In [27]:
# from lowrank_prune.lib.model_wrapper_low import *

In [28]:
for name, module in model.named_modules():
        if isinstance(module, ActLinear):
          print(name)
          module.record_activation = False
          module.clear_act_buffer()

model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.0.mlp.gate_proj
model.layers.0.mlp.up_proj
model.layers.0.mlp.down_proj
model.layers.1.self_attn.q_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.1.self_attn.o_proj
model.layers.1.mlp.gate_proj
model.layers.1.mlp.up_proj
model.layers.1.mlp.down_proj
model.layers.2.self_attn.q_proj
model.layers.2.self_attn.k_proj
model.layers.2.self_attn.v_proj
model.layers.2.self_attn.o_proj
model.layers.2.mlp.gate_proj
model.layers.2.mlp.up_proj
model.layers.2.mlp.down_proj
model.layers.3.self_attn.q_proj
model.layers.3.self_attn.k_proj
model.layers.3.self_attn.v_proj
model.layers.3.self_attn.o_proj
model.layers.3.mlp.gate_proj
model.layers.3.mlp.up_proj
model.layers.3.mlp.down_proj
model.layers.4.self_attn.q_proj
model.layers.4.self_attn.k_proj
model.layers.4.self_attn.v_proj
model.layers.4.self_attn.o_proj
model.layers.4.mlp.g

In [29]:
target_module_list = ['q_proj', 'v_proj']

In [30]:
rank = 16
rs = 32
n_iter = 30
divide_num = len(target_module_list)

In [31]:
model.device

device(type='cuda', index=0)

In [32]:
num_hidden_layers = model.config.num_hidden_layers
device=model.device
weight_list = {}
current_num = -1
for layer in range(num_hidden_layers):
    print(f"layer {layer}")
    layer_filter_fn = (
        lambda x: f"layers.{layer}." in x
    )  ### TODO # hack for llama series

    # enable recording for the current layer.
    for name, module in model.named_modules():
        if layer_filter_fn(name) and isinstance(module, ActLinear):
            # print("enabling recording for ", name)
            # if 'v_proj' in name or 'q_proj' in name:
            module.record_activation = True
            module.clear_act_buffer()

    for batch in dataloader_safe:
        inp, tar = batch[0].to(device), batch[1].to(device)
        assert True, "should run in disentangle mode"
        mask = tar.ne(-100)
        with set_mask(model, mask):
            model(inp)

    for name, module in model.named_modules():
        if layer_filter_fn(name) and isinstance(module, ActLinear):
            print("Module name:", name)
            # print(module.activation_norms)
            f_name = ""
            for t_name  in target_module_list:
                if t_name in name:
                    f_name = t_name
                    break
            if len(f_name)==0:
                # print("Wrong name ", name)
                continue
            current_num += 1
            layer_n =f_name
            print(layer_n)
            module.activation_norms = torch.cat(module.activation_norms, dim=0).to(
                device
            )  # size * d_in
            score = (
                module.activation_norms @ module.base.weight.data.T
            )  # (size * d_in) @ (d_out * d_in).T --> (size, d_out)
            d_out, d_in = module.base.weight.data.shape
            print(d_out, d_in)
            total_rank = min(d_out, d_in)
            print(
                f"remaining: rank {name} = {total_rank - rank} / {total_rank}"
            )
            # for removing from the top
            U, S, V = torch.svd_lowrank(
                score.float(), q=rs, niter=n_iter
            )  # (size, r) (r) (d_out, r)
            V =  V.type(module.base.weight.data.dtype)

            weight_list[layer_n+'_'+str(current_num//divide_num)+'_V'] = V


    for name, module in model.named_modules():
        if layer_filter_fn(name) and isinstance(module, ActLinear):
            # print("disabling recording for ", name)
            module.record_activation = False
            module.clear_act_buffer()

layer 0
Module name: model.layers.0.self_attn.q_proj
q_proj
4096 4096
remaining: rank model.layers.0.self_attn.q_proj = 4080 / 4096
Module name: model.layers.0.self_attn.k_proj
Module name: model.layers.0.self_attn.v_proj
v_proj
1024 4096
remaining: rank model.layers.0.self_attn.v_proj = 1008 / 1024
Module name: model.layers.0.self_attn.o_proj
Module name: model.layers.0.mlp.gate_proj
Module name: model.layers.0.mlp.up_proj
Module name: model.layers.0.mlp.down_proj
layer 1
Module name: model.layers.1.self_attn.q_proj
q_proj
4096 4096
remaining: rank model.layers.1.self_attn.q_proj = 4080 / 4096
Module name: model.layers.1.self_attn.k_proj
Module name: model.layers.1.self_attn.v_proj
v_proj
1024 4096
remaining: rank model.layers.1.self_attn.v_proj = 1008 / 1024
Module name: model.layers.1.self_attn.o_proj
Module name: model.layers.1.mlp.gate_proj
Module name: model.layers.1.mlp.up_proj
Module name: model.layers.1.mlp.down_proj
layer 2
Module name: model.layers.2.self_attn.q_proj
q_proj


In [33]:
weight_list.keys()

dict_keys(['q_proj_0_V', 'v_proj_0_V', 'q_proj_1_V', 'v_proj_1_V', 'q_proj_2_V', 'v_proj_2_V', 'q_proj_3_V', 'v_proj_3_V', 'q_proj_4_V', 'v_proj_4_V', 'q_proj_5_V', 'v_proj_5_V', 'q_proj_6_V', 'v_proj_6_V', 'q_proj_7_V', 'v_proj_7_V', 'q_proj_8_V', 'v_proj_8_V', 'q_proj_9_V', 'v_proj_9_V', 'q_proj_10_V', 'v_proj_10_V', 'q_proj_11_V', 'v_proj_11_V', 'q_proj_12_V', 'v_proj_12_V', 'q_proj_13_V', 'v_proj_13_V', 'q_proj_14_V', 'v_proj_14_V', 'q_proj_15_V', 'v_proj_15_V', 'q_proj_16_V', 'v_proj_16_V', 'q_proj_17_V', 'v_proj_17_V', 'q_proj_18_V', 'v_proj_18_V', 'q_proj_19_V', 'v_proj_19_V', 'q_proj_20_V', 'v_proj_20_V', 'q_proj_21_V', 'v_proj_21_V', 'q_proj_22_V', 'v_proj_22_V', 'q_proj_23_V', 'v_proj_23_V', 'q_proj_24_V', 'v_proj_24_V', 'q_proj_25_V', 'v_proj_25_V', 'q_proj_26_V', 'v_proj_26_V', 'q_proj_27_V', 'v_proj_27_V', 'q_proj_28_V', 'v_proj_28_V', 'q_proj_29_V', 'v_proj_29_V', 'q_proj_30_V', 'v_proj_30_V', 'q_proj_31_V', 'v_proj_31_V'])

In [34]:
weight_list['v_proj_29_V'].shape

torch.Size([1024, 32])

In [35]:
ds = 32

In [36]:
current_num = -1
# old_weights = {}
for layer in range(num_hidden_layers):

    print(f"layer {layer}")
    layer_filter_fn = (
        lambda x: f"layers.{layer}." in x
    )  ### TODO # hack for llama series

    # enable recording for the current layer.
    for name, module in model.named_modules():
        if layer_filter_fn(name) and isinstance(module, ActLinear):
            module.record_activation = True

    # forward pass and get activation records.

    # with torch.no_grad():
    for batch in dataloader:
        inp, tar = batch[0].to(device), batch[1].to(device)
        assert True, "should run in disentangle mode"
        mask = tar.ne(-100)
        with set_mask(model, mask):
            model(inp)

    for name, module in model.named_modules():
        if layer_filter_fn(name) and isinstance(module, ActLinear):
            print("Module name:", name)
            f_name = ""
            for t_name  in target_module_list:
                if t_name in name:
                    f_name = t_name
                    break
            if len(f_name)==0:
                # print("Wrong name ", name)
                continue
            current_num += 1
            layer_n = f_name
            print(layer_n)
            module.activation_norms = torch.cat(module.activation_norms, dim=0).to(
                device
            )  # size * d_in
            score = (
                module.activation_norms @ module.base.weight.data.T
            )  # (size * d_in) @ (d_out * d_in).T --> (size, d_out)
            d_out, d_in = module.base.weight.data.shape
            print(d_out, d_in)
            total_rank = min(d_out, d_in)
            print(
                f"remaining: rank {name} = {total_rank - rank} / {total_rank}"
            )

            # for removing from the bottom
            U, S, V = torch.svd_lowrank(
                score.float(), q=ds, niter=n_iter
            )  # (size, r) (r) (d_out, r)
            V =  V.type(module.base.weight.data.dtype)

            U2, S2, V2 = torch.svd_lowrank(
                module.base.weight.data.float(), q=rank, niter=n_iter
            )  # n(size, r) (r) (d_out, r)
            U2 =  U2.type(module.base.weight.data.dtype)
            S2 =  S2.type(module.base.weight.data.dtype)
            V2 =  V2.type(module.base.weight.data.dtype)

            safeV = weight_list[layer_n+'_'+str(current_num//divide_num)+'_V']
            weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_C'] = torch.eye(safeV.size(0),dtype=module.base.weight.dtype, device=module.base.weight.device) - safeV @ safeV.T

            weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_B'] = U2 @ torch.diag(torch.sqrt(S2))
            weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_B'] = V @ V.T @ weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_B']
            weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_A'] = torch.diag(torch.sqrt(S2)) @ V2.T

            # old_weights[layer_n+'_'+str(current_num//divide_num)+'og_weight'] = module.base.weight.data

            module.base.weight.data.sub_(
                weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_C'] @ weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_B'] @ weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_A'])
            weight_list[layer_n+'_'+str(current_num//divide_num)+'weight'] = module.base.weight.data
            # print(weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_B'].size(), weight_list[layer_n+'_'+str(current_num//divide_num)+'lora_A'].size())
        # torch.cuda.empty_cache()

# disable recording for the current layer.
    for name, module in model.named_modules():
        if layer_filter_fn(name) and isinstance(module, ActLinear):
            # print("disabling recording for ", name)
            module.record_activation = False
            module.clear_act_buffer()

layer 0
Module name: model.layers.0.self_attn.q_proj
q_proj
4096 4096
remaining: rank model.layers.0.self_attn.q_proj = 4080 / 4096
Module name: model.layers.0.self_attn.k_proj
Module name: model.layers.0.self_attn.v_proj
v_proj
1024 4096
remaining: rank model.layers.0.self_attn.v_proj = 1008 / 1024
Module name: model.layers.0.self_attn.o_proj
Module name: model.layers.0.mlp.gate_proj
Module name: model.layers.0.mlp.up_proj
Module name: model.layers.0.mlp.down_proj
layer 1
Module name: model.layers.1.self_attn.q_proj
q_proj
4096 4096
remaining: rank model.layers.1.self_attn.q_proj = 4080 / 4096
Module name: model.layers.1.self_attn.k_proj
Module name: model.layers.1.self_attn.v_proj
v_proj
1024 4096
remaining: rank model.layers.1.self_attn.v_proj = 1008 / 1024
Module name: model.layers.1.self_attn.o_proj
Module name: model.layers.1.mlp.gate_proj
Module name: model.layers.1.mlp.up_proj
Module name: model.layers.1.mlp.down_proj
layer 2
Module name: model.layers.2.self_attn.q_proj
q_proj


In [37]:
weight_list.keys()

dict_keys(['q_proj_0_V', 'v_proj_0_V', 'q_proj_1_V', 'v_proj_1_V', 'q_proj_2_V', 'v_proj_2_V', 'q_proj_3_V', 'v_proj_3_V', 'q_proj_4_V', 'v_proj_4_V', 'q_proj_5_V', 'v_proj_5_V', 'q_proj_6_V', 'v_proj_6_V', 'q_proj_7_V', 'v_proj_7_V', 'q_proj_8_V', 'v_proj_8_V', 'q_proj_9_V', 'v_proj_9_V', 'q_proj_10_V', 'v_proj_10_V', 'q_proj_11_V', 'v_proj_11_V', 'q_proj_12_V', 'v_proj_12_V', 'q_proj_13_V', 'v_proj_13_V', 'q_proj_14_V', 'v_proj_14_V', 'q_proj_15_V', 'v_proj_15_V', 'q_proj_16_V', 'v_proj_16_V', 'q_proj_17_V', 'v_proj_17_V', 'q_proj_18_V', 'v_proj_18_V', 'q_proj_19_V', 'v_proj_19_V', 'q_proj_20_V', 'v_proj_20_V', 'q_proj_21_V', 'v_proj_21_V', 'q_proj_22_V', 'v_proj_22_V', 'q_proj_23_V', 'v_proj_23_V', 'q_proj_24_V', 'v_proj_24_V', 'q_proj_25_V', 'v_proj_25_V', 'q_proj_26_V', 'v_proj_26_V', 'q_proj_27_V', 'v_proj_27_V', 'q_proj_28_V', 'v_proj_28_V', 'q_proj_29_V', 'v_proj_29_V', 'q_proj_30_V', 'v_proj_30_V', 'q_proj_31_V', 'v_proj_31_V', 'q_proj_0lora_C', 'q_proj_0lora_B', 'q_proj_0lora

In [38]:
# old_weights.keys()
# weight_list['q_proj_9lora_C']
# weight_list['q_proj_0lora_B']
# weight_list['q_proj_27weight']
# old_weights['q_proj_27og_weight']
# torch.equal(weight_list['q_proj_10weight'], old_weights['q_proj_10og_weight'])

# key = 'q_proj_27'
# weight_list[key+'lora_C'] @ weight_list[key+'lora_B'] @ weight_list[key+'lora_A']
# torch.equal(old_weights[key+'og_weight'] - weight_list[key+'lora_C'] @ weight_list[key+'lora_B'] @ weight_list[key+'lora_A'],\
#             old_weights[key+'og_weight'])

In [39]:
# ah nvm probably a reference pointer

In [40]:
model = revert_Act_to_Linear(model)
model.zero_grad()

Reverting model.layers.0.self_attn.q_proj, parent: model.layers.0.self_attn
Reverting model.layers.0.self_attn.k_proj, parent: model.layers.0.self_attn
Reverting model.layers.0.self_attn.v_proj, parent: model.layers.0.self_attn
Reverting model.layers.0.self_attn.o_proj, parent: model.layers.0.self_attn
Reverting model.layers.0.mlp.gate_proj, parent: model.layers.0.mlp
Reverting model.layers.0.mlp.up_proj, parent: model.layers.0.mlp
Reverting model.layers.0.mlp.down_proj, parent: model.layers.0.mlp
Reverting model.layers.1.self_attn.q_proj, parent: model.layers.1.self_attn
Reverting model.layers.1.self_attn.k_proj, parent: model.layers.1.self_attn
Reverting model.layers.1.self_attn.v_proj, parent: model.layers.1.self_attn
Reverting model.layers.1.self_attn.o_proj, parent: model.layers.1.self_attn
Reverting model.layers.1.mlp.gate_proj, parent: model.layers.1.mlp
Reverting model.layers.1.mlp.up_proj, parent: model.layers.1.mlp
Reverting model.layers.1.mlp.down_proj, parent: model.layers.

In [41]:
from peft import LoraConfig, get_peft_model

In [42]:
rank = 16

In [43]:
peft_config = LoraConfig(
        r = rank,
        lora_alpha = rank, # lora_alpha should match r to maintain scaling = 1
        lora_dropout = 0,
        inference_mode=False,
        init_lora_weights=False, # PiSSA initialization
        target_modules = target_module_list,
        task_type="CAUSAL_LM",
    )

In [44]:
model = get_peft_model(model, peft_config)

In [45]:
model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848


In [46]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_features=4096, out_fe

In [47]:
# # weight_list['q_proj_10lora_A'].shape
# for name, module in model.named_modules():
#     if "lora_" in name and "dropout" not in name:
#       try:
#         print(name, module['default'].weight.data.shape)
#       except:
#         try:
#           print(name, module.weight.data.shape)
#         except:
#           pass
# weight_list[key+'lora_B'].shape, weight_list['v_proj_12lora_B'].shape

In [48]:
# ok

In [49]:
module_counters = {t_name: 0 for t_name in target_module_list}

for name, module in model.named_modules():
    for t_name in target_module_list:
        if t_name in name:
            layer_idx = module_counters[t_name] // divide_num

            if 'lora_A.default' in name:
                key = f'{t_name}_{layer_idx}lora_A'
                if key in weight_list:
                    module.weight.data = weight_list[key]
                    print(f"Set {name} with {key}")
                else:
                    print(f"Warning: {key} not found")

            elif 'lora_B.default' in name:
                key = f'{t_name}_{layer_idx}lora_B'
                if key in weight_list:
                    module.weight.data = weight_list[key]
                    print(f"Set {name} with {key}")
                else:
                    print(f"Warning: {key} not found")

            elif t_name + '.' not in name:  # Main module (not lora_A or lora_B)
                key = f'{t_name}_{layer_idx}lora_C'
                if key in weight_list:
                    module.lora_C = weight_list[key].clone()
                    weight_list[key] = module.lora_C.clone()
                    module.requires_grad = False
                    print(f"Set {name} lora_C with {key}")
                else:
                    print(f"Warning: {key} not found")

                module_counters[t_name] += 1

            break
weight_list['divide_num'] = divide_num

Set base_model.model.model.layers.0.self_attn.q_proj lora_C with q_proj_0lora_C
Set base_model.model.model.layers.0.self_attn.q_proj.lora_A.default with q_proj_0lora_A
Set base_model.model.model.layers.0.self_attn.q_proj.lora_B.default with q_proj_0lora_B
Set base_model.model.model.layers.0.self_attn.v_proj lora_C with v_proj_0lora_C
Set base_model.model.model.layers.0.self_attn.v_proj.lora_A.default with v_proj_0lora_A
Set base_model.model.model.layers.0.self_attn.v_proj.lora_B.default with v_proj_0lora_B
Set base_model.model.model.layers.1.self_attn.q_proj lora_C with q_proj_0lora_C
Set base_model.model.model.layers.1.self_attn.q_proj.lora_A.default with q_proj_1lora_A
Set base_model.model.model.layers.1.self_attn.q_proj.lora_B.default with q_proj_1lora_B
Set base_model.model.model.layers.1.self_attn.v_proj lora_C with v_proj_0lora_C
Set base_model.model.model.layers.1.self_attn.v_proj.lora_A.default with v_proj_1lora_A
Set base_model.model.model.layers.1.self_attn.v_proj.lora_B.defa

In [50]:
model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848


In [51]:
save_folder = "/content/drive/MyDrive/CAISI_phase2/safe-finetuning/SaLoRA/outputs/debug_run"
os.makedirs(save_folder, exist_ok=True)

In [52]:
weight_list['divide_num'] = divide_num

# save_path = args.name+'_'+str(rank) + '_' + str(args.lr) + '_' + str(args.wd) + 'lora_ABC.pt'
save_path = os.path.join(save_folder, 'lora_ABC.pt')
# os.makedirs(os.path.dirname(save_path), exist_ok = True)
torch.save(weight_list, save_path)


In [53]:
import gc

In [54]:
del weight_list
# del old_weights
del dataloader
del dataloader_safe

gc.collect()

torch.cuda.empty_cache()

In [55]:
# # model.state_dict().keys()
# model.state_dict()['base_model.model.model.layers.24.self_attn.q_proj.lora_A.default.weight'].shape
# torch.cuda.empty_cache()
# del old_weights
# del dataloader
# del dataloader_safe
#  import gc
# gc.collect()
# torch.cuda.empty_cache()

In [56]:
# torch.equal(model.state_dict()['base_model.model.model.layers.24.self_attn.q_proj.lora_A.default.weight'], \
#             weight_list['q_proj_24lor`\a_A'])

In [57]:
dataset = datasets.load_from_disk('/content/drive/MyDrive/CAISI_phase2/safe-finetuning/ft_datasets/alpaca_clean_sample_5k')
train_ds = dataset['train']
val_ds = dataset['test']

def convert_to_template_from_alpaca(example):
    return {"prompt": [
                         {"role": "user", "content": example['instruction'] + ": " + example['input']}],
            "completion": [
                         {"role": "assistant", "content": example['output']}
                        ]
           }

og_cols = list(train_ds.column_names)
train_ds = train_ds.map(convert_to_template_from_alpaca, remove_columns=og_cols)
val_ds = val_ds.map(convert_to_template_from_alpaca, remove_columns=og_cols)

In [58]:
train_ds[10]

{'prompt': [{'content': 'Construct a Boolean expression to decide whether a number is divisible by 5.: ',
   'role': 'user'}],
 'completion': [{'content': 'A number is divisible by 5 if the remainder when the number is divided by 5 is equal to zero. We can use the modulo operator (%) to determine the remainder of a division. So, the boolean expression to decide whether a number `n` is divisible by 5 is `(n % 5) == 0`.',
   'role': 'assistant'}]}

In [59]:
config = Box.from_yaml(filename='/content/drive/MyDrive/CAISI_phase2/safe-finetuning/SaLoRA/configs/llama/config.yaml')

In [60]:
config

Box({'random_state': 5, 'attn_implementation': 'flash_attention_3', 'use_liger_kernel': False, 'rank': 16, 'rs': 32, 'ds': 32, 'target_modules': ['q_proj', 'v_proj'], 'data_path': '/content/drive/MyDrive/CAISI_phase2/safe-finetuning/ft_datasets/alpaca_clean_sample_5k', 'data_template': 'alpaca', 'base_model_path': 'meta-llama/Meta-Llama-3.1-8B-Instruct', 'dtype': 'float16', 'device': 'cuda', 'eval_strategy': 'epoch', 'save_strategy': 'epoch', 'logging_strategy': 'steps', 'logging_steps': 10, 'bs': 16, 'gradient_accumulation_steps': 1, 'use_gradient_checkpointing': True, 'use_packing': False, 'lr': 0.0002, 'wd': 0.01, 'optim': 'adamw_torch_fused', 'lr_scheduler_type': 'linear', 'warmup_ratio': 0.05, 'max_seq_length': 8192, 'num_epochs': 5})

In [61]:
model.get_nb_trainable_parameters()

(6815744, 8037076992)

In [62]:
training_args = SFTConfig(
        learning_rate=config.lr,
        lr_scheduler_type=config.lr_scheduler_type,
        use_liger_kernel=config.use_liger_kernel,
        warmup_ratio=config.warmup_ratio,
        weight_decay=config.wd,
        optim=config.optim,
        gradient_checkpointing=config.use_gradient_checkpointing,
        per_device_train_batch_size=config.bs,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        max_length=config.max_seq_length,
        packing=config.use_packing,
        num_train_epochs=2,
        save_only_model = False,
        eval_strategy = config.eval_strategy,
        save_strategy = config.save_strategy,
        logging_first_step=True,
        logging_strategy=config.logging_strategy,
        logging_steps=config.logging_steps,
        logging_dir=save_folder,
        log_level="info",
        seed=config.random_state,
        report_to = "none",
        output_dir=save_folder,
        eos_token=tokenizer.eos_token
    )

In [63]:
trainer = SFTTrainer(
        model=model, # peft model
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer
    )

The model is already on multiple devices. Skipping the move to device specified in `args`.
Using auto half precision backend


In [64]:
trainer.get_num_trainable_parameters()

6815744

In [65]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"]='expandable_segments:True'

In [66]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.
The following columns in the Training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: completion, prompt. If completion, prompt are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 4,250
  Num Epochs = 2
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 532
  Number of trainable parameters = 6,815,744


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.068900,1.052962,1.445642,819294.000000,0.692763
2,1.000900,1.043117,1.380601,1638588.000000,0.694883


The following columns in the Evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: completion, prompt. If completion, prompt are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 750
  Batch size = 8
Saving model checkpoint to /content/drive/MyDrive/CAISI_phase2/safe-finetuning/SaLoRA/outputs/debug_run/checkpoint-266
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Meta-Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659/config.json
Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_

TrainOutput(global_step=532, training_loss=1.0789871262876611, metrics={'train_runtime': 1306.1469, 'train_samples_per_second': 6.508, 'train_steps_per_second': 0.407, 'total_flos': 1.7190272401956864e+17, 'train_loss': 1.0789871262876611, 'epoch': 2.0})

In [67]:
! nvidia-smi

Fri Oct 31 03:54:41 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   51C    P0             91W /  400W |   70857MiB /  81920MiB |      1%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----